In [1]:
import os
import requests
from bs4 import BeautifulSoup
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import cdsapi
import xarray as xr
from statsmodels.tsa.stattools import grangercausalitytests
import rasterio
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from typing import Optional, Dict, Any
import pandas as pd
import sys
from pyproj import Transformer

In [2]:
NORTH_URL = "https://noaadata.apps.nsidc.org/NOAA/G02135/north/daily/geotiff/"
DOWNLOAD_DIR = "data/geotiff_images"
ANALYSIS_DIR = "data/analysis_results"

In [3]:
def projected_to_latlon(x, y, src_crs):
    transformer = Transformer.from_crs(src_crs, "EPSG:4326", always_xy=True)
    lon, lat = transformer.transform(x, y)
    return lat, lon

In [ ]:
class SatelliteImageFetcher:
    def __init__(self, base_url, download_dir):
        self.base_url = base_url
        self.download_dir = download_dir 
    
    def get_folders(self, url):
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        
        folders = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if href.endswith('/') and href != "../":
                folders.append(href)
                
        return folders
    
    def get_files(self, url):
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")
        
        tif_files = []
        for a in soup.find_all('a', href=True):
            filename = a['href']
            
            if (filename.endswith(".tif") and filename.startswith("N_") and "concentration_v4.0" in filename):
                tif_files.append(filename)
                
        return tif_files 
    
    def download_file(self, file_url, save_dir):
        filename = file_url.split('/')[-1]
        save_path = os.path.join(save_dir, filename)
        
        if os.path.exists(save_path):
            return f"Skipped (exists): {filename}"
        
        os.makedirs(save_dir, exist_ok=True)
        
        response = requests.get(file_url)
        with open(save_path, 'wb') as f:
            f.write(response.content)
            
        return f"Downloaded: {filename}"
    
    def fetch(self, year_range=None, months=None):
        print("Getting year folders...")
        year_folders = self.get_folders(self.base_url)
        print(f"Found {len(year_folders)} folders")
        
        downloaded = 0
        skipped = 0
        
        for year_folder in year_folders:
            year_str = year_folder.strip('/')
            
            if not (year_str.isdigit() and len(year_str) == 4):
                continue 
            
            year = int(year_str)
            
            # Filter years
            if year_range is not None:
                start, end = year_range
                if not (start <= year <= end):
                    continue
            
            print(f"\nYear {year}...")
            year_url = self.base_url + year_folder
            
            month_folders = self.get_folders(year_url)
            
            for month_folder in month_folders:
                month_url = year_url + month_folder 
                month_num = month_folder.split('_')[0]
                
                tif_files = self.get_files(month_url)
                
                if tif_files:
                    print(f"    {month_folder.strip('/')}: {len(tif_files)} files")
                
                for tif_file in tif_files:
                    file_url = month_url + tif_file
                    save_dir = os.path.join(self.download_dir, str(year), month_num)
                    
                    result = self.download_file(file_url, save_dir)
                    
                    if "Downloaded" in result:
                        downloaded += 1
                    else:
                        skipped += 1
            
        return {
            "downloaded": downloaded,
            "skipped": skipped,
            "total": downloaded + skipped
        }

In [5]:
fetch_data = SatelliteImageFetcher(NORTH_URL, DOWNLOAD_DIR)
result = fetch_data.fetch(year_range=(2025, 2025), months=None)
print(result)

Getting year folders...
Found 48 folders

Year 2025...
    01_Jan: 31 files
    02_Feb: 28 files
    03_Mar: 31 files
    04_Apr: 30 files
    05_May: 31 files
    06_Jun: 30 files
    07_Jul: 31 files
    08_Aug: 31 files
    09_Sep: 30 files
    10_Oct: 31 files
    11_Nov: 30 files
    12_Dec: 15 files
{'downloaded': 3, 'skipped': 346, 'total': 349}


In [ ]:
ERA5_VARIABLES = {
    "u10": "10m_u_component_of_wind", # east west
    "v10": "10m_v_component_of_wind", # north south
    "t2m": "2m_temperature",
    "msl": "mean_sea_level_pressure",
    "sst": "sea_surface_temperature",
    "tp": "total_precipitation",
    "mwd": "mean_wave_direction",
    "mwp": "mean_wave_period",
    "swh": "significant_height_of_combined_wind_waves_and_swell"
}


class EnvironmentalDataFetcher:
    """
    Fetch environmental variables from ECMWF ERA5
    """
    
    def download_era5_data(self, year, month, variable, region):
        """
        Download ERA5 data for specified variable
        Region format: [north, west, south, east]
        """
        atmos_var = ERA5_VARIABLES.get(variable, None)
        
        atmos_file = f"ERA5_atmos_{variable}_{year}_{month:02d}.nc"
        merged_file = f"ERA5_merged_{variable}_{year}_{month:02d}.nc"
        
        # If file exists, validate it
        if os.path.exists(atmos_file):
            try:
                xr.open_dataset(atmos_file).close()
                print(f"[FOUND] Atmospheric file already exists: {atmos_file}")
                return atmos_file
            except Exception:
                print(f"[CORRUPT] Removing invalid NetCDF: {atmos_file}")
                os.remove(atmos_file)
        
        client = cdsapi.Client()
        
        # ---------------------------
        # DOWNLOAD ATMOS
        # ---------------------------
        if atmos_var:
            if not os.path.exists(atmos_file):
                print(f"[DOWNLOADING] Atmospheric data ({variable}) for {year}-{month:02d}")
                
                dataset = "reanalysis-era5-single-levels"
                request = {
                    "product_type": "reanalysis",
                    "variable": atmos_var,
                    "year": str(year),
                    "month": f"{month:02d}",
                    "day": [f"{d:02d}" for d in range(1, 32)],
                    "time": [f"{h:02d}:00" for h in range(24)],
                    "area": region,
                    "format": "netcdf"
                }
                target = atmos_file 
                
                client.retrieve(dataset, request, target)
                print(f"[DONE] Atmospheric file saved: {atmos_file}")
            else:
                print(f"[FOUND] Atmospheric file already exists: {atmos_file}")
        else:
            print(f"[SKIP] No atmospheric variable found for: {variable}")
        
        return atmos_file
        
    def load_daily_data(self, nc_path, var_name):
        """
        Opens hourly ERA5 file and computes daily mean
        """
        ds = xr.open_dataset(nc_path)
        
        if "time" not in ds and "valid_time" in ds:
            ds = ds.rename({"valid_time": "time"})
            
        if var_name == "tp":
            daily = ds.resample(time="1D").sum()
        else:
            daily = ds.resample(time="1D").mean()
            
        daily["time"] = daily.time.dt.floor("D")

        return daily
    
    def interpolate_to_grid(self, ds, target_lat, target_lon):
        """
        Interpolate ERA5 dataset to match satellite image grid.
        """
        
        # Ensure latitude/longitude dim names are correct
        if ("latitude" not in ds.coords) and ("lat" in ds.coords):
            ds = ds.rename({"lat": "latitude"})
        if ("longitude" not in ds.coords) and ("lon" in ds.coords):
            ds = ds.rename({"lon": "longitude"})
        
        # Ensure ERA5 has correct time coordinate
        if "time" not in ds and "valid_time" in ds:
            ds = ds.rename({"valid_time": "time"})
            
        target_lat = np.asarray(target_lat)
        target_lon = np.asarray(target_lon)    
        
        # Make sure target_lat/lon are 2D
        if target_lat.ndim == 2 and target_lon.ndim == 2:
            lat2d = target_lat
            lon2d = target_lon
        # Case B: user passes 1D latitude and 1D longitude
        elif target_lat.ndim == 1 and target_lon.ndim == 1:
            lon2d, lat2d = np.meshgrid(target_lon, target_lat)
        else:
            raise ValueError(
                f"Invalid target_lat/target_lon shapes: "
                f"{target_lat.shape}, {target_lon.shape}"
            )
            
        
        # ---- ACTUAL INTERPOLATION ----
        return ds.interp(
            latitude=(("y", "x"), lat2d),
            longitude=(("y", "x"), lon2d),
            method="nearest"
        )   

In [ ]:
class SeaIceDataLoader:
    def __init__(self, base_dir: str):
        self.base_dir = Path(base_dir)

    def _build_filename(self, date: datetime) -> str:
        # pattern: N_YYYYMMDD_concentration_v4.0.tif
        return f"N_{date.strftime('%Y%m%d')}_concentration_v4.0.tif"

    def get_latest_image(
        self,
        ref_date: datetime | None = None,
        max_lookback: int = 14,
        timezone: str = "Asia/Jakarta",
        normalize: bool = False
    ):
    
        if ref_date is None:
            tz = ZoneInfo(timezone)
            ref_date = datetime.now(tz).replace(hour=0, minute=0, second=0, microsecond=0)

        current_date = ref_date

        for _ in range(max_lookback + 1):

            year_folder = current_date.strftime("%Y")
            month_folder = current_date.strftime("%m")

            folder = self.base_dir / year_folder / month_folder

            filename = f"N_{current_date.strftime('%Y%m%d')}_concentration_v4.0.tif"
            fpath = folder / filename

            if fpath.exists():
                print(f"[FOUND] Using file: {fpath}")

                with rasterio.open(fpath) as src:
                    img = src.read(1)

                    h, w = img.shape
                    rows, cols = np.indices((h, w))

                    lon, lat = rasterio.transform.xy(src.transform, rows, cols)
                    lat = np.array(lat).reshape(h,w)
                    lon = np.array(lon).reshape(h,w)

                return {
                    "date": current_date,
                    "path": fpath,
                    "image": img,
                    "lat": lat,
                    "lon": lon
                }

            current_date -= timedelta(days=1)

        print(f"[WARN] No GeoTIFF found for last {max_lookback} days.")
        return None

In [ ]:
GEOTIFF_DIR = "data/geotiff_images"    
MAX_LOOKBACK_DAYS = 30                  
ERA5_REGION = [90, -180, 60, 180]       # N, W, S, E
ERA5_MONTHS_TO_DOWNLOAD = list(range(1,13))      

print("\n" + "="*70)
print("LOADING LATEST SEA ICE IMAGE (REAL-TIME MODE)")
print("="*70)

# 1) Initialize loader and get the latest available image (today or prior)
loader = SeaIceDataLoader(GEOTIFF_DIR)
latest = loader.get_latest_image(max_lookback=MAX_LOOKBACK_DAYS, normalize=False)
print("FIRST LAT SHAPE:", latest["lat"].shape)
print("FIRST LON SHAPE:", latest["lon"].shape)

if latest is None:
    raise RuntimeError(f"No recent GeoTIFF found in {GEOTIFF_DIR} (lookback={MAX_LOOKBACK_DAYS})")

print(f"\n✓ Using satellite file: {latest['path'].name}")
print(f"✓ Image date: {latest['date'].strftime('%Y-%m-%d')}")
print(f"✓ Image shape: {latest['image'].shape}")

with rasterio.open(latest['path']) as src:
    height, width = src.height, src.width
    cols, rows = np.meshgrid(
        np.arange(width),
        np.arange(height),
        indexing = "ij"
    )

    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
    xs = np.array(xs)
    ys = np.array(ys)

    lat, lon = projected_to_latlon(xs, ys, src.crs)

    latest["lat"] = lat.reshape(height, width)
    latest["lon"] = lon.reshape(height, width)

print("✓ Forced reprojection to lat/lon")

print("latest lat shape:", latest["lat"].shape)
print("latest lon shape:", latest["lon"].shape)

print("SAT LAT min:", np.nanmin(latest["lat"]))
print("SAT LAT max:", np.nanmax(latest["lat"]))
print("SAT LON min:", np.nanmin(latest["lon"]))
print("SAT LON max:", np.nanmax(latest["lon"]))


HIST_DAYS = 90  
historical_images = []

if HIST_DAYS > 0:
    latest_date = latest['date']
    for d in range(HIST_DAYS):
        dt = latest_date - timedelta(days=(HIST_DAYS - 1 - d))
        candidate = loader.get_latest_image(ref_date=dt, max_lookback=0, normalize=False)
        if candidate is not None:
            historical_images.append(candidate)
    print(f"\n✓ Collected {len(historical_images)} historical images (last {HIST_DAYS} days)")

dates_to_cover = sorted({latest['date']} | {itm['date'] for itm in historical_images})

min_date = min(dates_to_cover) - timedelta(days=5)
max_date = max(dates_to_cover) + timedelta(days=5)

print("\n" + "="*70)
print("DOWNLOAD & INTERPOLATE ERA5 (daily) FOR REQUIRED PERIOD")
print("="*70)

env_fetcher = EnvironmentalDataFetcher()

era5_interpolated = {}  

for var, nc_name in ERA5_VARIABLES.items():
    print(f"\n[INFO] Preparing ERA5 variable: {var}")

    month_filepaths = []
    cur = datetime(min_date.year, min_date.month, 1)
    end = datetime(max_date.year, max_date.month, 1)
    
    while cur <= end:
        try:
            nc_fp = env_fetcher.download_era5_data(
                year=cur.year, 
                month=cur.month, 
                variable=var, 
                region=ERA5_REGION
            )
            month_filepaths.append(nc_fp)
        except Exception as e:
            print(f"[WARN] Failed to download ERA5 {var} {cur.year}-{cur.month:02d}: {e}")
        if cur.month == 12:
            cur = datetime(cur.year + 1, 1, 1)
        else:
            cur = datetime(cur.year, cur.month + 1, 1)

    daily_list = []
    for fp in month_filepaths:
        if fp is None:
            continue
        ds_daily = env_fetcher.load_daily_data(fp, var) 
        daily_list.append(ds_daily)
    if len(daily_list) == 0:
        print(f"[ERROR] No ERA5 data available for {var} in the requested period.")
        continue

    ds_all = xr.concat(daily_list, dim='time')
    print("ERA5 lat min:", float(ds_all.latitude.min()))
    print("ERA5 lat max:", float(ds_all.latitude.max()))
    print("ERA5 lon min:", float(ds_all.longitude.min()))
    print("ERA5 lon max:", float(ds_all.longitude.max()))    
    

    target_lat = latest['lat']
    target_lon = latest['lon']
    
    print("latest lat shape:", latest["lat"].shape)
    print("latest lon shape:", latest["lon"].shape)
    interp = env_fetcher.interpolate_to_grid(ds_all, target_lat, target_lon)

    if isinstance(interp, xr.Dataset):
        var_names = list(interp.data_vars)
        
        if nc_name in interp.data_vars:
            arr = interp[nc_name]
        else:
            arr = interp[var_names[0]]
    else:
        arr = interp  

    era5_interpolated[var] = ArithmeticError

env_data = {}

time_values = era5_interpolated[list(era5_interpolated.keys())[0]].time.values

for t in time_values:
    date_only = np.datetime64(t, "D")
    date_str = str(date_only)
    env_data[date_str] = {}

    for var, arr in era5_interpolated.items():
        try:
            sel = arr.sel(
                time=date_only,
                method="nearest"
            )

            time_diff = np.abs(sel.time.values - date_only)
            if time_diff > np.timedelta64(2, "D"):
                raise ValueError("ERA5 time too far from target date")

            env_data[date_str][var] = sel.values

        except Exception as e:
            env_data[date_str][var] = None

print("\n" + "="*70)
print("ALIGN ERA5 & SATELLITE AND BUILD COMBINED DATASET")
print("="*70)

combined_dataset = []

all_images = historical_images + [latest] if HIST_DAYS > 0 else [latest]

for item in all_images:
    date = item['date']
    date_np = np.datetime64(date.date())
    env_for_date = {}
    missing_vars = []
    for var, arr in era5_interpolated.items():
        try:
            sel = arr.sel(time=date_np, method="nearest")
            
            time_diff = np.abs(sel.time.values - date_np)
            if time_diff > np.timedelta64(1, "D"):
                raise ValueError("Nearest ERA5 time too far from SIC date")
            
            env_for_date[var] = sel.values
        except Exception as e:
            print(f"[ALIGN FAIL] {var} on {date_np}: {e}")
            env_for_date[var] = None
            missing_vars.append(var)
            
    combined_dataset.append({
        "date": date,
        "sic_image": item['image'],
        "lat": item['lat'],
        "lon": item['lon'],
        "env": env_for_date,
        "missing_env": missing_vars
    })

print("\nENV availability check (first sample):")
for var, val in combined_dataset[0]["env"].items():
    print(var, "OK" if val is not None else "MISSING")

print(f"\n✓ Combined dataset entries: {len(combined_dataset)}")
print("Sample entry keys:", combined_dataset[0].keys())
if len(combined_dataset[0]['missing_env']) > 0:
    print("Note: some environmental variables missing for certain dates; handle these before training (interp/drop).")

out_dir = Path("data/processed_data")
out_dir.mkdir(exist_ok=True)
import pickle
with open(out_dir / f"combined_latest_snapshot_{latest['date'].strftime('%Y%m%d')}.pkl", "wb") as f:
    pickle.dump(combined_dataset, f)
print(f"\n✓ Saved snapshot to {out_dir}")

print("\nREADY: You can now build sequences from combined_dataset for D+1/D+3/D+5 prediction.")



LOADING LATEST SEA ICE IMAGE (REAL-TIME MODE)
[FOUND] Using file: data\geotiff_images\2025\12\N_20251215_concentration_v4.0.tif
FIRST LAT SHAPE: (448, 304)
FIRST LON SHAPE: (448, 304)

✓ Using satellite file: N_20251215_concentration_v4.0.tif
✓ Image date: 2025-12-15
✓ Image shape: (448, 304)
✓ Forced reprojection to lat/lon
latest lat shape: (448, 304)
latest lon shape: (448, 304)
SAT LAT min: 31.10267175243087
SAT LAT max: 89.8368159996151
SAT LON min: -180.0
SAT LON max: 179.8139753954925
[FOUND] Using file: data\geotiff_images\2025\09\N_20250917_concentration_v4.0.tif
[FOUND] Using file: data\geotiff_images\2025\09\N_20250918_concentration_v4.0.tif
[FOUND] Using file: data\geotiff_images\2025\09\N_20250919_concentration_v4.0.tif
[FOUND] Using file: data\geotiff_images\2025\09\N_20250920_concentration_v4.0.tif
[FOUND] Using file: data\geotiff_images\2025\09\N_20250921_concentration_v4.0.tif
[FOUND] Using file: data\geotiff_images\2025\09\N_20250922_concentration_v4.0.tif
[FOUND] Us

In [16]:
import pickle

with open("data/processed_data/combined_latest_snapshot_20251215.pkl", "rb") as f:
    data = pickle.load(f)

print("Total entries:", len(data))
print("Keys:", data[0].keys())

print("\nEnvironment variables:", data[0]["env"].keys())
print("\nMissing env:", data[0]["missing_env"])

print("\nSIC image shape:", data[0]["sic_image"].shape)

Total entries: 91
Keys: dict_keys(['date', 'sic_image', 'lat', 'lon', 'env', 'missing_env'])

Environment variables: dict_keys(['u10', 'v10', 't2m', 'msl', 'sst', 'tp', 'mwd', 'mwp', 'swh'])

Missing env: []

SIC image shape: (448, 304)
